<a href="https://colab.research.google.com/github/jayasinghesasmitha/EN4554-Deep-Learning-For-Vision/blob/CNN-assignment-2/220261V_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN4554 Assignment 2 – CNNs (PyTorch)

**Name:** Jayasinghe J.A.S.U.  **Index No.:** 220261V
**GitHub:** https://github.com/jayasinghesasmitha/EN4554-Deep-Learning-For-Vision.git

**Settings used throughout:** CIFAR-10, 45k train / 5k validation split (fixed seed), per-channel normalisation with *training* statistics, seed = 42. The test set is touched exactly once, in Task 4.

In [1]:
import math, time, random, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
USE_AMP = (device == 'cuda')
QUICK = False
E = lambda n: 1 if QUICK else n
print('device:', device, '| AMP:', USE_AMP, '| torch', torch.__version__)

device: cuda | AMP: True | torch 2.11.0+cu130


## Data: CIFAR-10, 45k/5k split, normalisation with training statistics

In [2]:
tv_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
Xall = torch.from_numpy(tv_train.data).permute(0, 3, 1, 2).float() / 255.0
yall = torch.tensor(tv_train.targets)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(50000, generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]
Xtr, ytr, Xva, yva = Xall[tr_idx], yall[tr_idx], Xall[va_idx], yall[va_idx]

mean = Xtr.mean(dim=(0, 2, 3), keepdim=True)
std  = Xtr.std(dim=(0, 2, 3), keepdim=True)
Xtr, Xva = (Xtr - mean) / std, (Xva - mean) / std
Xtr, ytr, Xva, yva = Xtr.to(device), ytr.to(device), Xva.to(device), yva.to(device)
DATA = (Xtr, ytr, Xva, yva)
print(Xtr.shape, Xva.shape, 'mean', mean.flatten().tolist(), 'std', std.flatten().tolist())

100%|██████████| 170M/170M [47:56<00:00, 59.3kB/s]


torch.Size([45000, 3, 32, 32]) torch.Size([5000, 3, 32, 32]) mean [0.49165481328964233, 0.4823332130908966, 0.4466792047023773] std [0.24708116054534912, 0.2435086965560913, 0.2615806758403778]
